Tarea: Con el modelo general de YOLO detectar en el vídeo proporcionado vehículos y personas

Clases para flujo

In [1]:
class Tracked_Object:

    def __init__(self, id : int, type : int) -> None:
        self.id = id
        self.points = [] # los distintos puntos asociados a este objeto
        self.trayectory_x = 0
        self.trayectory_y = 0
        self.type = type

    def add_point(self, xy) -> None:
        x, y = xy[0], xy[1]
        self.points.append((x, y))

    def calculate_trayectory(self) -> None:
        self.trayectory_x = self.points[-1][0] - self.points[0][0]
        self.trayectory_y = self.points[-1][1] - self.points[0][1]

    def __str__(self) -> str:
        return f"ID:{self.id}, type:{self.type}"

class Image_Section: 

    def __init__(self, section_name : str) -> None:
        self.name = section_name
        self.class_names = {0:"person", 1:"bicycle", 2:"car", 3:"motorbike", 5:"bus", 7:"truck"}
        self.detections = {} # par clave-valor clase : cuenta

    def add_obj(self, obj : Tracked_Object) -> None:
        self.detections[obj.type] = self.detections.get(obj.type, 0) + 1

    def __str__(self):
        total = sum(self.detections.values())
        string = f"Esta sección de {self.name} contiene {total} objetos:\n"
        for key, count in self.detections.items():
            string += f"- {self.class_names.get(key, f'clase_{key}')} : {count}\n"
        return string



Detección y seguimiento de personas, vehículos y matrículas - Volcado de vídeo

In [ ]:
from paddleocr import PaddleOCR
import cv2  
from ultralytics import YOLO

#video
filename = "coches.MP4"
ANCHO_MAX = 1280

#clases que se van a detectar
classCounter = {"person": 0, "bicycle": 0, "car": 0, "motorbike": 0, "bus": 0, "truck": 0}
classNames = {0: "person", 1: "bicycle", 2: "car", 3: "motorbike", 5: "bus", 7: "truck"}
uniqueObjects = {nombre: set() for nombre in classNames.values()}

#carga de los modelos
yolo_model = YOLO('yolo11n.pt')
yolo_mat_model = YOLO('YOLO_best_640.pt')
ocr = PaddleOCR(use_angle_cls=True, lang='es', enable_mkldnn=False)
last_known_position = {} 
objects = {} # clave-par id:obj
coches = cv2.VideoCapture(filename)

# Obtener propiedades del vídeo original
fps = coches.get(cv2.CAP_PROP_FPS)
width = int(coches.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(coches.get(cv2.CAP_PROP_FRAME_HEIGHT))


# Calcular resolución de salida
if width > ANCHO_MAX:
    factor = ANCHO_MAX / width
    output_width = ANCHO_MAX
    output_height = int(height * factor)
else:
    output_width = width
    output_height = height

# Crear el vídeo de salida
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    "resultado.mp4",
    fourcc,
    fps,
    (output_width, output_height)
)

if not out.isOpened():
    print("Error al crear el vídeo de salida")

while(coches.isOpened()):
    ret, frame = coches.read()

    if not ret:
        break


    alto, ancho = frame.shape[:2]
    if ancho > ANCHO_MAX:
        factor = ANCHO_MAX / ancho
        frame = cv2.resize(frame, (ANCHO_MAX, int(alto * factor)))
    alto, ancho = frame.shape[:2] 

    # Reiniciar contadores en cada frame
    classCounter = {nombre: 0 for nombre in classNames.values()}   

    clean_frame = frame.copy()

    #entrenar el modelo
    results_yolo = yolo_model.track(frame, persist=True, classes=[0,1,2,3,5,7], verbose=False)
    results_yolo_mat = yolo_mat_model.track(frame, persist=True, verbose=False)

    frame = results_yolo[0].plot()
    frame = results_yolo_mat[0].plot(img=frame)

    mat_boxes = results_yolo_mat[0].boxes

    if mat_boxes is not None and len(mat_boxes) > 0:
        coordenadas = mat_boxes.xyxy.cpu().numpy()

        for i, box in enumerate(coordenadas):

            x1, y1, x2, y2 = map(int, box)

            # Asegurar que las coordenadas están dentro de la imagen
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(ancho, x2)
            y2 = min(alto, y2)

            # Recortar la región correspondiente de clean_frame
            matricula = clean_frame[y1:y2, x1:x2]

            # Evitar recortes vacíos
            if matricula.size == 0:
                continue

            # Pasar el recorte a PaddleOCR
            result_paddle = ocr.predict(matricula)

            if result_paddle:
                textos = result_paddle[0].get("rec_texts", [])

                texto = " ".join(textos)
                if texto and len(texto) >= 7:
                    print("Matrícula detectada:", texto)

                    cv2.putText(
                        frame,
                        texto,
                        (x1, max(y1 - 30, 40)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2
                    ) 

    #recorrer resultados
    for result in results_yolo:
        for box in result.boxes:
            clase = int(box.cls.item())

            if clase not in classNames:
                continue

            nombre = classNames[clase]
            classCounter[nombre] += 1

            if box.id is not None:
                id_objeto = int(box.id.item())
                uniqueObjects[nombre].add(id_objeto)
        
        ids = result.boxes.id.int().cpu().tolist()
        boxes_xywh = result.boxes.xywh.cpu().tolist()
        types = result.boxes.cls.int().cpu().tolist()

        for id, coords, type in zip(ids, boxes_xywh, types):
            if id not in objects:
                objects[id] = Tracked_Object(id, type)
            objects[id].add_point(coords)

    #Dibujar contador
    objetos = []

    for nombre in classNames.values():
        actual = classCounter[nombre]
        total = len(uniqueObjects[nombre])

        objetos.append(f"{nombre}: {actual} (Total: {total})")


    fuente = cv2.FONT_HERSHEY_SIMPLEX
    escala = 0.55
    grosor = 2
    margen = 10
    separacion = 25

    # Calcular dimensiones del panel
    dimensiones = [
        cv2.getTextSize(texto, fuente, escala, grosor)[0]
        for texto in objetos
    ]

    ancho_panel = max(w for w, h in dimensiones) + 20
    alto_panel = len(objetos) * separacion + 15

    # Situar panel en esquina superior derecha
    x_panel = max(0, ancho - ancho_panel - margen)
    y_panel = margen

    # Dibujar fondo negro
    cv2.rectangle(
        frame,
        (x_panel, y_panel),
        (x_panel + ancho_panel, y_panel + alto_panel),
        (0, 0, 0),
        -1
    )

    # Dibujar cada lÃnea
    for i, texto in enumerate(objetos):

        x = x_panel + 10
        y = y_panel + 20 + i * separacion

        cv2.putText(
            frame,
            texto,
            (x, y),
            fuente,
            escala,
            (0, 255, 0),
            grosor,
            cv2.LINE_AA
        )

    out.write(frame)
    cv2.imshow('Vid', frame)


    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

coches.release()
out.release()
cv2.destroyAllWindows()

print("Vídeo guardado correctamente")

lu = Image_Section("arriba izquierda")
ld = Image_Section("abajo izquierda")
ru = Image_Section("arriba derecha")
rd = Image_Section("abajo derecha")

for id in objects.keys():
    obj = objects[id]
    obj.calculate_trayectory()

    if obj.trayectory_x < 0 and obj.trayectory_y < 0:
        ld.add_obj(obj)
    elif obj.trayectory_x < 0 and obj.trayectory_y > 0:
        lu.add_obj(obj)
    elif obj.trayectory_x > 0 and obj.trayectory_y > 0:
        ru.add_obj(obj)
    else:
        rd.add_obj(obj)

print(objects)

print(lu)
print(ld)
print(ru)
print(rd)



Volcado de CSV

In [ ]:
from paddleocr import PaddleOCR
import cv2  
from ultralytics import YOLO
import csv

#video
filename = "coches.MP4"
ANCHO_MAX = 1280



#clases que se van a detectar
classCounter = {"person": 0, "bicycle": 0, "car": 0, "motorbike": 0, "bus": 0, "truck": 0}
classNames = {0: "person", 1: "bicycle", 2: "car", 3: "motorbike", 5: "bus", 7: "truck"}
uniqueObjects = {nombre: set() for nombre in classNames.values()}

#carga de los modelos
yolo_model = YOLO('yolo11n.pt')
yolo_mat_model = YOLO('YOLO_best_640.pt')
ocr = PaddleOCR(use_angle_cls=True, lang='es', enable_mkldnn=False)

#Creación del archivo CSV
csv_file = open("detecciones.csv", "w", newline="", encoding="utf-8-sig")

campos = [
    "fotograma",
    "tipo_objeto",
    "confianza_objeto",
    "identificador_tracking",
    "x1", "y1", "x2", "y2",
    "matricula_en_su_caso",
    "confianza_matricula",
    "mx1", "my1", "mx2", "my2",
    "texto_matricula"
]

writer = csv.DictWriter(csv_file, fieldnames=campos)
writer.writeheader()

numero_fotograma = 0

coches = cv2.VideoCapture(filename)

while(True):
    ret, frame = coches.read()

    if not ret:
        break


    alto, ancho = frame.shape[:2]
    if ancho > ANCHO_MAX:
        factor = ANCHO_MAX / ancho
        frame = cv2.resize(frame, (ANCHO_MAX, int(alto * factor)))
    alto, ancho = frame.shape[:2] 

    # Reiniciar contadores en cada frame
    classCounter = {nombre: 0 for nombre in classNames.values()}   

    clean_frame = frame.copy()

    #entrenar el modelo
    results_yolo = yolo_model.track(frame, persist=True, classes=[0,1,2,3,5,7], verbose=False)
    results_yolo_mat = yolo_mat_model.track(frame, persist=True, verbose=False)

    frame = results_yolo[0].plot()
    frame = results_yolo_mat[0].plot(img=frame)

    mat_boxes = results_yolo_mat[0].boxes

    matriculas_detectadas = []

    if mat_boxes is not None and len(mat_boxes) > 0:
        coordenadas = mat_boxes.xyxy.cpu().numpy()
        confianzas = mat_boxes.conf.cpu().numpy()

        for i, box in enumerate(coordenadas):

            mx1, my1, mx2, my2 = map(int, box)

            # Asegurar que las coordenadas están dentro de la imagen
            mx1 = max(0, mx1)
            my1 = max(0, my1)
            mx2 = min(ancho, mx2)
            my2 = min(alto, my2)

            # Recortar la región correspondiente de clean_frame
            matricula = clean_frame[my1:my2, mx1:mx2]

            # Evitar recortes vacíos
            if matricula.size == 0:
                continue

            # Pasar el recorte a PaddleOCR
            result_paddle = ocr.predict(matricula)

            if result_paddle:
                textos = result_paddle[0].get("rec_texts", [])
                texto = " ".join(textos)

                matriculas_detectadas.append({
                    "confianza": float(confianzas[i]),
                    "mx1": mx1,
                    "my1": my1,
                    "mx2": mx2,
                    "my2": my2,
                    "texto": texto
                })

                if texto and len(texto) >= 7:
                    print("Matrícula detectada:", texto)

                    cv2.putText(
                        frame,
                        texto,
                        (x1, max(y1 - 30, 40)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2
                    )
    #---------------------------------------------------------------------------
    result = results_yolo[0]
    boxes = result.boxes

    if boxes is not None and len(boxes) > 0:

        coordenadas_obj = boxes.xyxy.cpu().numpy()
        clases = boxes.cls.cpu().numpy()
        confianzas_obj = boxes.conf.cpu().numpy()

        # Obtener identificadores de tracking
        if boxes.id is not None:
            ids = boxes.id.int().cpu().numpy()
        else:
            ids = [None] * len(boxes)

        for i, box in enumerate(coordenadas_obj):

            x1, y1, x2, y2 = map(int, box)

            clase = int(clases[i])
            nombre = classNames[clase]

            confianza_obj = float(confianzas_obj[i])
            id_objeto = ids[i]

            # Buscar matrículas cuyo centro esté dentro del objeto
            matricula_asociada = None

            for mat in matriculas_detectadas:

                centro_x = (mat["mx1"] + mat["mx2"]) / 2
                centro_y = (mat["my1"] + mat["my2"]) / 2

                if x1 <= centro_x <= x2 and y1 <= centro_y <= y2:

                    # Si hay varias, quedarse con la de mayor confianza
                    if (
                        matricula_asociada is None
                        or mat["confianza"] > matricula_asociada["confianza"]
                    ):
                        matricula_asociada = mat

            # Preparar información de matrícula
            if matricula_asociada is not None:

                tiene_matricula = "Sí"

                confianza_mat = matricula_asociada["confianza"]

                mx1 = matricula_asociada["mx1"]
                my1 = matricula_asociada["my1"]
                mx2 = matricula_asociada["mx2"]
                my2 = matricula_asociada["my2"]

                if len(matricula_asociada["texto"]) >= 7:
                    texto_matricula = matricula_asociada["texto"]
                else:
                    texto_matricula = "None"

            else:

                tiene_matricula = "No"

                confianza_mat = ""
                mx1 = ""
                my1 = ""
                mx2 = ""
                my2 = ""
                texto_matricula = ""

            # Escribir fila CSV
            writer.writerow({
                "fotograma": numero_fotograma,
                "tipo_objeto": nombre,
                "confianza_objeto": confianza_obj,
                "identificador_tracking": id_objeto if id_objeto is not None else "",
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": tiene_matricula,
                "confianza_matricula": confianza_mat,
                "mx1": mx1,
                "my1": my1,
                "mx2": mx2,
                "my2": my2,
                "texto_matricula": texto_matricula
            }) 


    
    #----------------------------------------------------------------------------
    #recorrer resultados
    for result in results_yolo:
        for box in result.boxes:
            clase = int(box.cls.item())

            if clase not in classNames:
                continue

            nombre = classNames[clase]
            classCounter[nombre] += 1

            if box.id is not None:
                id_objeto = int(box.id.item())
                uniqueObjects[nombre].add(id_objeto)

    #Dibujar contador
    objetos = []

    for nombre in classNames.values():
        actual = classCounter[nombre]
        total = len(uniqueObjects[nombre])

        objetos.append(f"{nombre}: {actual} (Total: {total})")


    fuente = cv2.FONT_HERSHEY_SIMPLEX
    escala = 0.55
    grosor = 2
    margen = 10
    separacion = 25

    # Calcular dimensiones del panel
    dimensiones = [
        cv2.getTextSize(texto, fuente, escala, grosor)[0]
        for texto in objetos
    ]

    ancho_panel = max(w for w, h in dimensiones) + 20
    alto_panel = len(objetos) * separacion + 15

    # Situar panel en esquina superior derecha
    x_panel = max(0, ancho - ancho_panel - margen)
    y_panel = margen

    # Dibujar fondo negro
    cv2.rectangle(
        frame,
        (x_panel, y_panel),
        (x_panel + ancho_panel, y_panel + alto_panel),
        (0, 0, 0),
        -1
    )

    # Dibujar cada lÃnea
    for i, texto in enumerate(objetos):

        x = x_panel + 10
        y = y_panel + 20 + i * separacion

        cv2.putText(
            frame,
            texto,
            (x, y),
            fuente,
            escala,
            (0, 255, 0),
            grosor,
            cv2.LINE_AA
        )

    numero_fotograma += 1
    cv2.imshow('Vid', frame)

# 
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

coches.release()
csv_file.close()
cv2.destroyAllWindows()